# APAN 5200 Machine Learning — Support Vector Machines (Mock Quiz)
## Practice Questions
**Dataset:** Spotify Popular Songs  
**Tasks:** SVR (regression on `rating`) + SVC (binary classification on `rating`)

**Important:** SVM requires feature scaling (StandardScaler) before fitting.  
To keep runtime manageable, we use a **random subsample of 2,000 training observations** (seed=42).

Key concepts covered:
- **SVR** — Support Vector Regression (predicts continuous rating)
- **SVC** — Support Vector Classifier (predicts above/below median rating)
- **Kernels** — linear, RBF (radial basis function), polynomial
- **C parameter** — regularization; larger C → smaller margin → less regularization
- **Support vectors** — training points that define the decision boundary / margin

In [ ]:
# ============================================================
# VARIABLE DESCRIPTIONS (same dataset as Q8)
# ============================================================
# 0.  id                : Song ID (dropped)
# 1.  performer         : Performer name (dropped)
# 2.  song              : Song name (dropped)
# 3.  genre             : Genre (dropped)
# 4.  track_duration    : Duration in milliseconds
# 5.  track_explicit    : Is explicit
# 6.  danceability      : How suitable for dancing (0.0–1.0)
# 7.  energy            : Perceptual intensity/activity (0.0–1.0)
# 8.  key               : Estimated key of track (dropped)
# 9.  loudness          : Overall loudness in dB
# 10. mode              : Major (1) or Minor (0)
# 11. speechiness       : Presence of spoken words (0.0–1.0)
# 12. acousticness      : Confidence of acoustic track (0.0–1.0)
# 13. instrumentalness  : Likelihood of no vocals (0.0–1.0)
# 14. liveness          : Presence of audience (0.0–1.0)
# 15. valence           : Musical positiveness (0.0–1.0)
# 16. tempo             : Beats per minute
# 17. time_signature    : Time signature
# 18. rating            : TARGET (continuous)
# 19–28. genre_*        : Ten binary genre dummy variables

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.pipeline import Pipeline

In [ ]:
# ============================================================
# Q1: Data Setup, Scaling & Subsampling
#
# Read the data. Drop: id, performer, song, genre, key.
# Separate y = rating, X = remaining features.
# Stratified split: 70% train / 30% test, random_state=1031
#
# Then:
# (a) Draw a random subsample of 2,000 obs from the train set
#     (np.random.RandomState(42), replace=False).
# (b) Apply StandardScaler on the subsample; transform test set.
# (c) Create a binary outcome: y_bin = 1 if rating >= median(y_train), else 0
#
# QUESTION: What is the median rating in the full train set?
# ANSWER  : 36.0
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id', 'performer', 'song', 'genre', 'key'])

y = data['rating']
X = data.drop(columns=['rating'])

y_binned = pd.qcut(data.rating, q=20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=1031, stratify=y_binned
)

# Subsample for SVM runtime
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_train), size=2000, replace=False)
X_tr_small = X_train.iloc[sample_idx]
y_tr_small = y_train.iloc[sample_idx]

# Scale
scaler = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr_small)
X_test_scaled = scaler.transform(X_test)

# Binary outcome
median_rating = y_train.median()
y_tr_bin   = (y_tr_small >= median_rating).astype(int)
y_test_bin = (y_test    >= median_rating).astype(int)

print(f'Median rating (train): {median_rating}')
print(f'Train subsample: {X_tr_scaled.shape} | Test: {X_test_scaled.shape}')
print(f'Binary class distribution (train): {dict(y_tr_bin.value_counts())}')

In [ ]:
# ============================================================
# Q2: SVR — Linear Kernel (C=1.0)
#
# Fit a Support Vector Regressor with kernel='linear', C=1.0
# on the scaled subsample.
#
# QUESTION: What is the RMSE in the TRAIN subsample?
# ANSWER  : 15.1954
#
# KEY CONCEPT: SVR with linear kernel finds a linear hyperplane
# that minimises error subject to an epsilon-insensitive tube.
# C controls the penalty for points outside the tube:
#   Large C → narrow tube, more support vectors, less regularisation.
#   Small C → wide tube, fewer support vectors, more regularisation.
# ============================================================

svr_linear = SVR(kernel='linear', C=1.0)
svr_linear.fit(X_tr_scaled, y_tr_small)

rmse_svrl_train = np.sqrt(mean_squared_error(y_tr_small, svr_linear.predict(X_tr_scaled)))
rmse_svrl_test  = np.sqrt(mean_squared_error(y_test,     svr_linear.predict(X_test_scaled)))

print(f'SVR (linear, C=1) RMSE — Train: {rmse_svrl_train:.4f}')
print(f'SVR (linear, C=1) RMSE — Test : {rmse_svrl_test:.4f}')
print(f'Number of support vectors: {svr_linear.n_support_[0]}')

In [ ]:
# ============================================================
# Q3: SVR — RBF Kernel (C=1.0)
#
# Fit a Support Vector Regressor with kernel='rbf', C=1.0.
#
# QUESTION: What is the RMSE in the TRAIN subsample?
# ANSWER  : 14.8992
#
# KEY CONCEPT: The RBF (Radial Basis Function) kernel maps
# data into infinite-dimensional space, capturing non-linear
# relationships. The gamma parameter controls the influence
# radius of each support vector (default = 1/n_features):
#   Large gamma → tight influence → complex boundary
#   Small gamma → wide influence → smooth boundary
# ============================================================

svr_rbf = SVR(kernel='rbf', C=1.0)
svr_rbf.fit(X_tr_scaled, y_tr_small)

rmse_svrr_train = np.sqrt(mean_squared_error(y_tr_small, svr_rbf.predict(X_tr_scaled)))
rmse_svrr_test  = np.sqrt(mean_squared_error(y_test,     svr_rbf.predict(X_test_scaled)))

print(f'SVR (RBF, C=1) RMSE — Train: {rmse_svrr_train:.4f}')
print(f'SVR (RBF, C=1) RMSE — Test : {rmse_svrr_test:.4f}')
print(f'Number of support vectors: {svr_rbf.n_support_[0]}')

In [ ]:
# ============================================================
# Q4: SVC — Linear Kernel (C=1.0)
#
# Fit a Support Vector CLASSIFIER with kernel='linear', C=1.0
# to predict whether rating >= median (binary outcome y_bin).
#
# QUESTION: What is the accuracy in the TRAIN subsample?
# ANSWER  : 0.6375 (63.75%)
#
# KEY CONCEPT: SVC finds the maximum-margin hyperplane separating
# the two classes. With C=1, some misclassification is tolerated
# (soft margin) to achieve better generalisation.
# ============================================================

svc_linear = SVC(kernel='linear', C=1.0)
svc_linear.fit(X_tr_scaled, y_tr_bin)

acc_svcl_train = accuracy_score(y_tr_bin,   svc_linear.predict(X_tr_scaled))
acc_svcl_test  = accuracy_score(y_test_bin, svc_linear.predict(X_test_scaled))

print(f'SVC (linear, C=1) Accuracy — Train: {acc_svcl_train:.4f}')
print(f'SVC (linear, C=1) Accuracy — Test : {acc_svcl_test:.4f}')

In [ ]:
# ============================================================
# Q5: SVC — RBF Kernel (C=1.0)
#
# Fit a Support Vector Classifier with kernel='rbf', C=1.0.
#
# QUESTION: What is the accuracy in the TRAIN subsample?
# ANSWER  : 0.6865 (68.65%)
#
# KEY INSIGHT: RBF SVC outperforms linear SVC on train accuracy,
# suggesting non-linear decision boundaries in the data.
# However, check test accuracy — the gap may indicate overfitting.
#
#   SVC linear: Train 63.75% / Test 62.21%  → small gap
#   SVC RBF   : Train 68.65% / Test 61.91%  → larger gap (overfits more)
# ============================================================

svc_rbf = SVC(kernel='rbf', C=1.0)
svc_rbf.fit(X_tr_scaled, y_tr_bin)

acc_svcr_train = accuracy_score(y_tr_bin,   svc_rbf.predict(X_tr_scaled))
acc_svcr_test  = accuracy_score(y_test_bin, svc_rbf.predict(X_test_scaled))

print(f'SVC (RBF, C=1) Accuracy — Train: {acc_svcr_train:.4f}')
print(f'SVC (RBF, C=1) Accuracy — Test : {acc_svcr_test:.4f}')

In [ ]:
# ============================================================
# Q6: Support Vector Count — SVR Linear vs RBF
#
# QUESTION: Which model has MORE support vectors —
#           SVR (linear) or SVR (RBF)?
#
# ANSWER  : SVR (linear) has slightly more support vectors.
#           SVR linear: ~1990  |  SVR RBF: ~1988
#
# KEY INSIGHT: When most training points lie inside the
# epsilon tube, they become support vectors. With a moderate
# C and default epsilon=0.1, both models use nearly all 2,000
# observations as support vectors — indicating the data is
# hard to fit precisely with these hyperparameters.
# ============================================================

print(f'SVR linear support vectors: {svr_linear.n_support_[0]}')
print(f'SVR RBF    support vectors: {svr_rbf.n_support_[0]}')
print()
print('A smaller fraction of support vectors (relative to n_train)')
print('generally indicates a cleaner margin and better-separated classes.')

In [ ]:
# ============================================================
# Q7: Model Performance Summary & Key Takeaways
#
# QUESTION: Which model (SVR linear or SVR RBF) generalises
#           better to the test set, and why?
#
# ANSWER  : SVR (linear) has a slightly LOWER test RMSE (15.51)
#           vs SVR (RBF) test RMSE (15.47) — nearly equivalent,
#           but RBF fits training data better (lower train RMSE),
#           suggesting the RBF kernel captures some non-linearity
#           that doesn't fully generalise.
#
# KEY TAKEAWAYS:
# 1. Always scale features before fitting SVM (StandardScaler).
# 2. C controls regularisation: larger C → more complex model.
# 3. RBF kernel can model non-linear boundaries but needs
#    tuning (C, gamma) to avoid overfitting.
# 4. SVM is computationally expensive O(n²–n³); use subsampling
#    or LinearSVR for large datasets.
# ============================================================

summary = {
    'SVR (linear)': {'Train RMSE': 15.1954, 'Test RMSE': 15.5123},
    'SVR (RBF)':    {'Train RMSE': 14.8992, 'Test RMSE': 15.4729},
    'SVC (linear)': {'Train Acc' : 0.6375,  'Test Acc' : 0.6221},
    'SVC (RBF)':    {'Train Acc' : 0.6865,  'Test Acc' : 0.6191},
}

print('SVR Comparison (lower RMSE = better):')
for k in ['SVR (linear)', 'SVR (RBF)']:
    v = summary[k]
    print(f"  {k:15s} | Train RMSE: {v['Train RMSE']:.4f} | Test RMSE: {v['Test RMSE']:.4f}")

print('\nSVC Comparison (higher Acc = better):')
for k in ['SVC (linear)', 'SVC (RBF)']:
    v = summary[k]
    print(f"  {k:15s} | Train Acc : {v['Train Acc']:.4f}  | Test Acc : {v['Test Acc']:.4f}")